In [ ]:
# Cell 1: Mount Drive and Peek at the Data
from google.colab import drive
import pandas as pd

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the path to your newly uploaded MU-IoT file
# UPDATE THIS PATH to match exactly where you uploaded it
mu_iot_file_path = "/content/drive/MyDrive/Colab_Datasets/Test_For_Model/subsest1.csv"

print("\nReading just the first 5 rows to inspect the columns...")

# 3. Read only 5 rows to prevent RAM issues
peek_df = pd.read_csv(mu_iot_file_path, nrows=5)

# 4. Print all 121 column names as a list
print(list(peek_df.columns))

# 5. Display the first few rows
peek_df.head()

Mounted at /content/drive

Reading just the first 5 rows to inspect the columns...
['FPT', 'LPT', 'FD', 'FAMin', 'FAMean', 'FAMax', 'FAStd', 'FIMean', 'FIMin', 'FIMax', 'FIStd', 'DSCP_C', 'DSCP_M', 'DSCP_UV', 'DSCP_S', 'ECN_C', 'ECN_M', 'ECN_UV', 'ECN_S', 'FPC', 'TPackets', 'TBytes', 'SBytes', 'DBytes', 'PLM', 'FI', 'FIAT', 'BIAT', 'ForwardPC', 'BackwardPC', 'FlowR', 'FlowStd', 'FlowV', 'FH_M', 'BH_M', 'FAPS', 'BAPS', 'FBPP', 'FFD', 'Proto', 'SDuration', 'SPC', 'SBCount', 'BytesPS', 'PacketsPS', 'RTT', 'HL_Sum', 'HL_Mode', 'HL_mean', 'SYN_FC', 'FIN_FC', 'RST_FC', 'PSH_FC', 'URG_FC', 'ECE_FC', 'CWR_FC', 'TCP_HD', 'RCount', 'DACount', 'MPackets', 'FThroughput', 'BThroughput', 'FJitter', 'BJitter', 'FBurst', 'FEntropy', 'PIA_Var', 'PL', 'PEntropy', 'MP_Count', 'BP_Count', 'MGA_L', 'MGA_UL', 'SIT', 'STT', 'FIPD_Var', 'BIPD_Var', 'TCPWS_Sum', 'TCPWS_Mean', 'TCPWS_Mode', 'HTTPRM_L', 'HTTPRM_UL', 'HTTPRM_M', 'HTTPGetC', 'HTTPPutC', 'HTTPPostC', 'HTTPOptC', 'HTTP_SCL', 'HTTP_SCUL:', 'HTTP_SCM'

,FPT,LPT,FD,FAMin,FAMean,FAMax,FAStd,FIMean,FIMin,FIMax,...,MQTT_MTL,MQTT_MTUL,MQTT_MTM,RTSP_ML,RTSP_MUL,RTSP_MM,RTSP_SETUPC,label,category,type
0,0.505728,0.505728,0.000000,0.000000,0.000000,0.000000,0.000000,0.499991,0.500026,0.499965,...,0.0,0.0,0.544594,0.0,0.0,0.2,0.071429,0,4,12
1,0.232334,0.232334,0.000000,0.000000,0.000000,0.000000,0.000000,0.499991,0.500026,0.499965,...,0.0,0.0,0.544594,0.0,0.0,0.2,0.071429,0,0,9
2,0.281320,0.281320,0.000000,0.000000,0.000000,0.000000,0.000000,0.499991,0.500026,0.499965,...,0.0,0.0,0.544594,0.0,0.0,0.2,0.071429,0,0,10
3,0.499095,0.499094,0.000053,0.000001,0.000018,0.000039,0.000032,0.499988,0.500007,0.499978,...,0.0,0.0,0.544594,0.0,0.0,0.2,0.071429,0,4,7
4,0.286387,0.286387,0.000000,0.000000,0.000000,0.000000,0.000000,0.499991,0.500026,0.499965,...,0.0,0.0,0.544594,0.0,0.0,0.2,0.071429,0,0,10


In [ ]:
# Cell 2: Bulletproof Feature Mapping and Chunk Sampling
import pandas as pd
import numpy as np
import pickle
import os

# --- 1. CONFIGURE PATHS ---
# Using the exact spelling that worked in Cell 1
mu_iot_file_path = "/content/drive/MyDrive/Colab_Datasets/Test_For_Model/subsest1.csv"
model_artifacts_path = "/content/drive/MyDrive/Colab_Datasets/PreprocessedForproject"

print("Checking Google Drive connection and files...")

# DEBUGGING: List all files in the directory to ensure Colab can see them
if os.path.exists(model_artifacts_path):
    files_in_dir = os.listdir(model_artifacts_path)
    print(f"Success! Found {len(files_in_dir)} files in your artifacts folder.")
else:
    raise FileNotFoundError(f"Colab cannot see the folder: {model_artifacts_path}. Try remounting Drive!")

# Automatically find the scaler file regardless of capitalization
scaler_filename = next((f for f in files_in_dir if f.lower() == 'scaler.pkl'), None)

if not scaler_filename:
    raise FileNotFoundError("Could not find any file named scaler.pkl in that folder!")

print(f"Loading Scaler: {scaler_filename}...")
with open(os.path.join(model_artifacts_path, scaler_filename), 'rb') as f:
    scaler = pickle.load(f)

expected_features = list(scaler.feature_names_in_)

# --- 2. THE TRANSLATION DICTIONARY ---
rename_map = {
    'FD': 'Flow Duration',
    'Proto': 'Protocol',
    'ForwardPC': 'Total Fwd Packets',
    'BackwardPC': 'Total Backward Packets',
    'SBytes': 'Total Length of Fwd Packets',
    'DBytes': 'Total Length of Bwd Packets',
    'BytesPS': 'Flow Bytes/s',
    'PacketsPS': 'Flow Packets/s',
    'SYN_FC': 'SYN Flag Count',
    'FIN_FC': 'FIN Flag Count',
    'RST_FC': 'RST Flag Count',
    'PSH_FC': 'PSH Flag Count',
    'URG_FC': 'URG Flag Count',
    'ECE_FC': 'ECE Flag Count',
    'CWR_FC': 'CWE Flag Count'
}

# --- 3. MEMORY-SAFE CHUNK SAMPLING ---
print(f"Reading MU-IoT dataset in chunks and extracting random samples...")
chunk_size = 200000
sample_fraction = 0.02

sampled_chunks = []

for chunk_idx, chunk in enumerate(pd.read_csv(mu_iot_file_path, chunksize=chunk_size, low_memory=False)):
    chunk.rename(columns=rename_map, inplace=True)

    sample = chunk.sample(frac=sample_fraction, random_state=42)
    sampled_chunks.append(sample)

    if chunk_idx % 5 == 0:
        print(f"Processed chunk {chunk_idx}...")

raw_test_set = pd.concat(sampled_chunks, ignore_index=True)
print(f"\nSampling complete. New test dataset shape: {raw_test_set.shape}")

# --- 4. FEATURE ALIGNMENT & PREPROCESSING ---
print("Aligning features to match CICIDS2017 architecture...")

missing_features = [col for col in expected_features if col not in raw_test_set.columns]
if missing_features:
    print(f"\nWARNING - Filling {len(missing_features)} missing features with 0. Top 5 missing: {missing_features[:5]}")

for col in expected_features:
    if col not in raw_test_set.columns:
        raw_test_set[col] = 0

X_new_raw = raw_test_set[expected_features].copy()

# Ensure the target column is handled properly
if 'label' in raw_test_set.columns:
    y_new = raw_test_set['label'].values
else:
    print("Warning: Target label column not found. Creating empty labels.")
    y_new = np.zeros(len(X_new_raw))

X_new_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
X_new_raw.fillna(0, inplace=True)

for col in X_new_raw.columns:
    X_new_raw[col] = pd.to_numeric(X_new_raw[col], downcast='float')

# --- 5. SCALE USING SAVED SCALER ---
print("\nApplying pre-trained Standard Scaler...")
X_new_scaled = scaler.transform(X_new_raw)

# --- 6. SAVE FOR SELF-LEARNING NOTEBOOK ---
print("Saving final matrices to Google Drive...")
np.save(os.path.join(model_artifacts_path, 'MU_IoT_X_test_scaled.npy'), X_new_scaled)
np.save(os.path.join(model_artifacts_path, 'MU_IoT_y_test.npy'), y_new)
print("Done! Ready for the Zero-Day and Self-Learning Notebook.")

Checking Google Drive connection and files...
Success! Found 11 files in your artifacts folder.
Loading Scaler: scaler.pkl...
Reading MU-IoT dataset in chunks and extracting random samples...
Processed chunk 0...
Processed chunk 5...
Processed chunk 10...
Processed chunk 15...
Processed chunk 20...
Processed chunk 25...
Processed chunk 30...
Processed chunk 35...
Processed chunk 40...

Sampling complete. New test dataset shape: (174238, 106)
Aligning features to match CICIDS2017 architecture...

WARNING - Filling 46 missing features with 0. Top 5 missing: ['Destination Port', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']

Applying pre-trained Standard Scaler...
Saving final matrices to Google Drive...
Done! Ready for the Zero-Day and Self-Learning Notebook.
